# Phase 1 — Experiment 2: PaLM-style Parallel Decoder (Optimized)

**Architecture:** PaLM-style (Chowdhery et al., 2022) — Parallel Attention + FFN

Key difference from Experiment 1 (LLaMA-style):
- **Parallel block**: `x = x + attn(norm(x)) + ffn(norm(x))` instead of sequential
- PaLM showed this matches sequential quality with ~15% faster training
- Attention and FFN share a single pre-norm, run in parallel on the same input
- Everything else identical: GQA, SwiGLU, RMSNorm, RoPE, weight tying

**Optimizations applied (vs original notebook):**
- GPU guard: raises error early if no CUDA device detected
- Updated AMP API (`torch.amp` instead of deprecated `torch.cuda.amp`)
- `torch.compile()` for ~10-30% speedup (PyTorch 2.0+)
- Larger micro-batch (16) with fewer accum steps → same eff_batch, better GPU util
- `num_workers=2` (less overhead on single-GPU Kaggle)
- Handles `torch.compile` model wrapping for checkpoint save/load
- **Requires Kaggle GPU: Settings → Accelerator → GPU T4 x2**

In [ ]:
!pip -q install tqdm huggingface_hub
import os, sys, json, time, math, random, shutil, zipfile
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("Imports OK  |  PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    raise RuntimeError(
        "\n" + "="*60 + "\n"
        "  NO GPU DETECTED — training will be impossibly slow!\n"
        "  Go to: Settings → Accelerator → GPU T4 x2\n"
        "  Then restart the notebook.\n"
        + "="*60
    )

## 1 — Configuration (all hyperparameters in one place)

In [ ]:
@dataclass
class Config:
    # ── Model ──
    vocab_size: int      = 40_000
    d_model: int         = 768
    n_layers: int        = 19
    n_query_heads: int   = 12
    n_kv_heads: int      = 4
    d_ff: int            = 2048
    max_seq_len: int     = 512
    dropout: float       = 0.1
    weight_tying: bool   = True
    rope_theta: float    = 10000.0

    # ── Training ──
    lr: float            = 3e-4
    min_lr: float        = 3e-5
    warmup_steps: int    = 1000
    max_steps: int       = 30_000
    batch_size: int      = 16       # ← doubled from 8 (fits T4 16GB)
    grad_accum: int      = 2        # ← halved to keep eff_batch=32
    weight_decay: float  = 0.1
    max_grad_norm: float = 1.0
    betas: tuple         = (0.9, 0.95)
    use_amp: bool        = True

    # ── Data ──
    shares: dict         = None
    val_frac: float      = 0.02

    # ── Logging / checkpointing ──
    log_interval: int    = 100
    eval_interval: int   = 2000
    save_interval: int   = 5000
    patience: int        = 5
    seed: int            = 42

    # ── Generation ──
    gen_max_words: int        = 128
    gen_temperature: float    = 0.6
    gen_top_k: int            = 40
    gen_top_p: float          = 0.85
    gen_rep_penalty: float    = 1.2

    def __post_init__(self):
        if self.shares is None:
            NATURAL = {"High Courts": 0.551, "Supreme Court": 0.073,
                       "Tribunals": 0.335, "Acts": 0.041}
            ALPHA = 0.5
            raw = {s: p ** ALPHA for s, p in NATURAL.items()}
            t = sum(raw.values())
            self.shares = {s: round(v / t, 2) for s, v in raw.items()}
            diff = round(1.0 - sum(self.shares.values()), 2)
            self.shares[max(self.shares, key=self.shares.get)] += diff
        self.head_dim = self.d_model // self.n_query_heads
        self.n_groups = self.n_query_heads // self.n_kv_heads
        assert self.d_model % self.n_query_heads == 0
        assert self.n_query_heads % self.n_kv_heads == 0

    @property
    def eff_batch(self): return self.batch_size * self.grad_accum

CFG = Config()
HF_REPO  = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"
OUT       = Path("/kaggle/working")
CKPT_DIR  = OUT / "palm_checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
EXP_NAME = "palm"

print(f"Config: ~{CFG.d_model}d × {CFG.n_layers}L, seq_len={CFG.max_seq_len}")
print(f"eff_batch={CFG.eff_batch}, max_steps={CFG.max_steps}")
print(f"Tokens per step: {CFG.eff_batch * CFG.max_seq_len:,}")
print(f"Total tokens: ~{CFG.max_steps * CFG.eff_batch * CFG.max_seq_len / 1e9:.1f}B")
print(f"Mixture (α=0.5): {CFG.shares}")


## 2 — Load pre-tokenized data (from data-prep notebook output)

In [ ]:
# ═══ Copy pre-tokenized data + tokenizer from Kaggle input datasets ═══
# Before running: add datasets as Input:
#   1. BPE tokenizer output (contains bpe_40k.json + a1_tokenizer.py)
#   2. Data-prep output (contains train_arr.npy + val_arr.npy)
#   3. (Optional) Previous run output (contains checkpoints/best.pt for resume)

INPUT = Path("/kaggle/input")
OUT   = Path("/kaggle/working")
CKPT_DIR = OUT / "palm_checkpoints"; CKPT_DIR.mkdir(exist_ok=True)

# ── Find files with flexible glob (handles Kaggle's "(2)" renames) ──
tok_json = None
tok_module = None
train_npy = None
val_npy = None

for dataset_dir in sorted(INPUT.iterdir()):
    if not dataset_dir.is_dir(): continue
    for f in dataset_dir.rglob("*"):
        if f.name == "bpe_40k.json" and tok_json is None:
            tok_json = f
        if f.name == "train_arr.npy" and train_npy is None:
            train_npy = f
        if f.name == "val_arr.npy" and val_npy is None:
            val_npy = f
    for f in dataset_dir.rglob("a1_tokenizer*.py"):
        if tok_module is None:
            tok_module = f

for name, src in [("bpe_40k.json", tok_json), ("a1_tokenizer.py", tok_module),
                  ("train_arr.npy", train_npy), ("val_arr.npy", val_npy)]:
    if src is None:
        raise FileNotFoundError(
            f"'{name}' not found in /kaggle/input/.\\n"
            "Add the BPE tokenizer output AND data-prep output as Kaggle Datasets.")
    dst = OUT / name
    shutil.copy2(src, dst)
    print(f"  ✓ {name} ← {src}")

# ── Resume from checkpoint if available in input ──
for dataset_dir in sorted(INPUT.iterdir()):
    if not dataset_dir.is_dir(): continue
    for f in dataset_dir.rglob("best.pt"):
        shutil.copy2(f, CKPT_DIR / "best.pt")
        print(f"  ✓ best.pt ← {f}")
        break

# ── Load tokenizer ──
sys.path.insert(0, str(OUT))
from a1_tokenizer import BPETokenizer
tokenizer = BPETokenizer.load(str(OUT / "bpe_40k.json"))
CFG.vocab_size = tokenizer.vocab_size
print(f"\\nTokenizer: vocab_size={tokenizer.vocab_size}")

# ── Load pre-tokenized arrays ──
train_arr = np.load(OUT / "train_arr.npy")
val_arr   = np.load(OUT / "val_arr.npy")
print(f"Train: {len(train_arr):,} tokens ({train_arr.nbytes/1e6:.0f} MB)")
print(f"Val:   {len(val_arr):,} tokens ({val_arr.nbytes/1e6:.0f} MB)")

# ── Download test set ──
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
test_path = hf_hub_download(HF_REPO, TEST_FILE, repo_type="dataset", token=HF_TOKEN)
print(f"Test: {test_path}")

# ── Dataset & DataLoader ──
class BlockDataset(Dataset):
    def __init__(self, data, block_len):
        self.data = data; self.block_len = block_len
        self.n = max(0, (len(data) - 1) // block_len)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        i = idx * self.block_len
        x = torch.from_numpy(self.data[i:i+self.block_len].astype(np.int64))
        y = torch.from_numpy(self.data[i+1:i+self.block_len+1].astype(np.int64))
        return x, y

train_ds = BlockDataset(train_arr, CFG.max_seq_len)
val_ds   = BlockDataset(val_arr, CFG.max_seq_len)
train_dl = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                      num_workers=2, pin_memory=True, drop_last=True)

print(f"\\nTrain: {len(train_ds):,} blocks ({len(train_ds)*CFG.max_seq_len:,} tokens)")
print(f"Val:   {len(val_ds):,} blocks")
print(f"Steps/epoch ≈ {len(train_ds) // CFG.batch_size // CFG.grad_accum:,}")


## 6 — Model architecture (PaLM-style Parallel Decoder)

Decoder-only Transformer with **parallel attention + FFN blocks** (Chowdhery et al., 2022):

**Key difference from Experiment 1 (LLaMA sequential):**
- LLaMA: `x → norm1 → attn → +residual → norm2 → ffn → +residual` (sequential)
- **PaLM: `x → norm → [attn(norm_x) + ffn(norm_x)] → +residual` (parallel)**

The attention and FFN sub-layers share a **single RMSNorm** and operate on the
same normalized input in parallel. PaLM showed this matches sequential quality
while being ~15% faster to train (fewer sequential dependencies on accelerator).

All other components are identical to Experiment 1:
- **Grouped Query Attention (GQA)** — 12 query heads, 4 KV heads
- **Rotary Positional Embeddings (RoPE)**
- **RMSNorm** (pre-norm)
- **SwiGLU** feed-forward
- **KV cache** for generation
- **Weight tying** (embedding ↔ lm_head)

In [ ]:
# ── RMSNorm ──
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

# ── RoPE ──
def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rotated = torch.cat([-x2, x1], dim=-1)
    c_exp = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], dim=-1)
    s_exp = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], dim=-1)
    return x * c_exp + rotated * s_exp

# ── GQA ──
class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh = cfg.n_query_heads
        self.nkh = cfg.n_kv_heads
        self.ng  = cfg.n_groups
        self.hd  = cfg.head_dim
        self.dm  = cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.n_query_heads * cfg.head_dim, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.o = nn.Linear(cfg.n_query_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nqh, self.hd).transpose(1,2)
        k = self.k(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        v = self.v(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        q = apply_rope(q, cos, sin, offset)
        k = apply_rope(k, cos, sin, offset)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkh, self.ng, k.size(2), self.hd)
            k = k.reshape(B, self.nqh, -1, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkh, self.ng, v.size(2), self.hd)
            v = v.reshape(B, self.nqh, -1, self.hd)
        w = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        if mask is not None: w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out)), new_cache

# ── SwiGLU FFN ──
class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

# ── PaLM-style Parallel Transformer Block ──
# Key change: single norm, attn and ffn run in parallel on the same normed input
# x = x + attn(norm(x)) + ffn(norm(x))   [PaLM, Chowdhery et al. 2022]
# vs sequential: x = x + attn(norm1(x)); x = x + ffn(norm2(x))   [LLaMA]
class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm = RMSNorm(cfg.d_model)   # single shared norm (PaLM-style)
        self.attn = GQA(cfg)
        self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        nx = self.norm(x)
        h, cache = self.attn(nx, cos, sin, mask, kv_cache, offset)
        x = x + h + self.ffn(nx)           # parallel: both operate on same normed input
        return x, cache

# ── Full model ──
class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop     = nn.Dropout(cfg.dropout)
        self.layers   = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm     = RMSNorm(cfg.d_model)
        self.lm_head  = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.weight_tying:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len * 2, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def _causal_mask(self, T, device, S=None):
        if S is None: S = T
        rows = torch.arange(T, device=device).unsqueeze(1) + (S - T)
        cols = torch.arange(S, device=device).unsqueeze(0)
        mask = torch.where(cols <= rows, 0.0, float("-inf"))
        return mask.unsqueeze(0).unsqueeze(0)

    def forward(self, ids, targets=None, kv_caches=None, offset=0):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        mask = self._causal_mask(T, ids.device, T + offset)
        new_caches = []
        for i, layer in enumerate(self.layers):
            c = kv_caches[i] if kv_caches else None
            x, nc = layer(x, self.rope_cos, self.rope_sin, mask, c, offset)
            new_caches.append(nc)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size),
                                   targets.view(-1), ignore_index=-1)
        return logits, loss, new_caches

    def count_params(self):
        return sum(p.numel() for p in self.parameters())

    @torch.no_grad()
    def generate(self, ids, max_new=256, temp=0.8, top_k=50, top_p=0.9,
                 rep_pen=1.1, stop_id=None):
        self.eval()
        B, T = ids.shape
        caches = None
        logits, _, caches = self.forward(ids, kv_caches=caches, offset=0)
        gen = ids
        off = T
        for _ in range(max_new):
            nl = logits[:, -1, :]
            if rep_pen != 1.0:
                for b in range(B):
                    for tid in set(gen[b].tolist()):
                        if nl[b, tid] > 0: nl[b, tid] /= rep_pen
                        else:              nl[b, tid] *= rep_pen
            if temp > 0:
                nl = nl / temp
                if top_k > 0:
                    tv, _ = torch.topk(nl, min(top_k, nl.size(-1)))
                    nl[nl < tv[:, -1:]] = float("-inf")
                if top_p < 1.0:
                    sl, si = torch.sort(nl, descending=True)
                    cp = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
                    mask = (cp - F.softmax(sl, dim=-1)) >= top_p
                    sl[mask] = float("-inf")
                    nl = sl.scatter(1, si, sl)
                probs = F.softmax(nl, dim=-1)
                nxt = torch.multinomial(probs, 1)
            else:
                nxt = nl.argmax(dim=-1, keepdim=True)
            gen = torch.cat([gen, nxt], dim=1)
            if stop_id is not None and (nxt == stop_id).all(): break
            if gen.size(1) >= self.cfg.max_seq_len: break
            logits, _, caches = self.forward(nxt, kv_caches=caches, offset=off)
            off += 1
        return gen

model = LegalLM(CFG).to(DEVICE)
n_params = model.count_params()
print(f"PaLM-style Model: {n_params:,} parameters ({n_params/1e6:.1f}M)")
assert 50_000_000 <= n_params <= 300_000_000, f"Outside 50M–300M range: {n_params:,}"

# ── torch.compile for extra speed (PyTorch 2.0+) ──
if hasattr(torch, "compile"):
    print("Applying torch.compile()...")
    model = torch.compile(model)
    print("  ✓ torch.compile applied")
else:
    print("  torch.compile not available (PyTorch < 2.0)")

# ── Smoke test ──
_x = torch.randint(0, CFG.vocab_size, (2, 32), device=DEVICE)
_l, _loss, _ = model(_x, targets=_x)
print(f"Smoke test: logits={_l.shape}, loss={_loss.item():.4f}")
_g = model.generate(_x[:1, :8], max_new=16, temp=1.0, top_k=10)
print(f"Generation smoke test: {_g.shape}")
del _x, _l, _loss, _g
torch.cuda.empty_cache()
print("✓ PaLM-style Model OK")

## 7 — Training loop

In [ ]:
from dataclasses import asdict

def get_lr(step):
    if step < CFG.warmup_steps:
        return CFG.lr * (step + 1) / CFG.warmup_steps
    if step >= CFG.max_steps:
        return CFG.min_lr
    prog = (step - CFG.warmup_steps) / max(1, CFG.max_steps - CFG.warmup_steps)
    cos_val = 0.5 * (1.0 + math.cos(math.pi * prog))
    return CFG.min_lr + (CFG.lr - CFG.min_lr) * cos_val

@torch.no_grad()
def evaluate(model, dl, max_batches=50):
    model.eval()
    tot = n = 0
    for i, (x, y) in enumerate(dl):
        if i >= max_batches: break
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=CFG.use_amp):
            _, loss, _ = model(x, targets=y)
        tot += loss.item(); n += 1
    model.train()
    return tot / max(n, 1)

def save_ckpt(model, opt, scaler, step, val_loss, path, cfg=None):
    """Atomic save: write to a temp file, then rename (rename is atomic on Linux).
    This prevents corrupted checkpoints if Kaggle kills the session mid-write.
    Also saves config so later phases can reconstruct the model."""
    # Unwrap compiled model if needed
    raw = model._orig_mod if hasattr(model, "_orig_mod") else model
    tmp_path = str(path) + ".tmp"
    payload = {"step": step, "val_loss": val_loss,
               "model": raw.state_dict(),
               "opt": opt.state_dict(),
               "scaler": scaler.state_dict()}
    if cfg is not None:
        payload["config"] = asdict(cfg)
    torch.save(payload, tmp_path)
    os.replace(tmp_path, path)  # atomic on Linux
    print(f"  💾 Saved {path}")

# ── Metrics collection ──
metrics = {
    "steps": [], "train_loss": [], "train_ppl": [],
    "learning_rate": [], "grad_norm": [], "tok_per_sec": [],
    "val_steps": [], "val_loss": [], "val_ppl": [],
}

# ── Build optimizer (unwrap compiled model for param iteration) ──
raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
decay, no_decay = [], []
for name, p in raw_model.named_parameters():
    if not p.requires_grad: continue
    if p.ndim < 2 or "norm" in name or "bias" in name:
        no_decay.append(p)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW([
    {"params": decay,    "weight_decay": CFG.weight_decay},
    {"params": no_decay, "weight_decay": 0.0},
], lr=CFG.lr, betas=CFG.betas)

scaler = torch.amp.GradScaler("cuda", enabled=CFG.use_amp)

# ── Resume? ──
start_step = 0
best_val = float("inf")
# Try best.pt first, then fall back to latest step_*.pt
resume = CKPT_DIR / "best.pt"
if not resume.exists():
    step_ckpts = sorted(CKPT_DIR.glob("step_*.pt"),
                        key=lambda p: int(p.stem.split("_")[1]), reverse=True)
    if step_ckpts:
        resume = step_ckpts[0]
        print(f"best.pt not found, falling back to {resume.name}")

if resume.exists():
    try:
        ckpt = torch.load(resume, map_location=DEVICE)
        raw_model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["opt"])
        scaler.load_state_dict(ckpt["scaler"])
        start_step = ckpt["step"]
        best_val = ckpt.get("val_loss", float("inf"))
        if "config" in ckpt:
            print(f"  Config found in checkpoint: d_model={ckpt['config']['d_model']}, "
                  f"n_layers={ckpt['config']['n_layers']}")
        print(f"Resumed from step {start_step}, best_val={best_val:.4f}")
    except RuntimeError as e:
        print(f"⚠ Checkpoint {resume} is corrupted: {e}")
        print("  Starting training from scratch.")
        start_step = 0
        best_val = float("inf")
        raw_model.apply(raw_model._init)

print(f"\nTraining from step {start_step} to {CFG.max_steps}...")

In [ ]:
train_start = time.time()   # ← track total training time

model.train()
train_iter = iter(train_dl)
running = 0.0
t0 = time.time()
no_improve = 0          # early-stopping patience counter

for step in range(start_step, CFG.max_steps):
    lr = get_lr(step)
    for pg in optimizer.param_groups: pg["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    acc_loss = 0.0

    for _ in range(CFG.grad_accum):
        try: x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_dl)
            x, y = next(train_iter)
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=CFG.use_amp):
            _, loss, _ = model(x, targets=y)
            loss = loss / CFG.grad_accum
        scaler.scale(loss).backward()
        acc_loss += loss.item()

    scaler.unscale_(optimizer)
    gn = nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
    scaler.step(optimizer)
    scaler.update()
    running += acc_loss

    if (step + 1) % CFG.log_interval == 0:
        avg = running / CFG.log_interval
        ppl = math.exp(min(avg, 20))
        el = time.time() - t0
        tps = CFG.log_interval * CFG.eff_batch * CFG.max_seq_len / el
        print(f"  step {step+1:6d}/{CFG.max_steps} | loss {avg:.4f} | "
              f"ppl {ppl:.1f} | lr {lr:.2e} | gn {gn:.2f} | "
              f"tok/s {tps:.0f} | {el:.0f}s")
        # ── Collect metrics ──
        metrics["steps"].append(step + 1)
        metrics["train_loss"].append(round(avg, 4))
        metrics["train_ppl"].append(round(ppl, 1))
        metrics["learning_rate"].append(round(lr, 8))
        metrics["grad_norm"].append(round(float(gn), 4))
        metrics["tok_per_sec"].append(round(tps, 0))
        running = 0.0; t0 = time.time()

    if (step + 1) % CFG.eval_interval == 0:
        vl = evaluate(model, val_dl)
        vp = math.exp(min(vl, 20))
        print(f"  >>> val loss={vl:.4f}  ppl={vp:.1f}")
        # ── Collect val metrics ──
        metrics["val_steps"].append(step + 1)
        metrics["val_loss"].append(round(vl, 4))
        metrics["val_ppl"].append(round(vp, 1))
        if vl < best_val:
            best_val = vl
            no_improve = 0
            save_ckpt(model, optimizer, scaler, step+1, vl, CKPT_DIR/"best.pt", cfg=CFG)
        else:
            no_improve += 1
            print(f"  ⚠ No improvement for {no_improve}/{CFG.patience} evals")
            if no_improve >= CFG.patience:
                print(f"\n🛑 Early stopping at step {step+1} (patience={CFG.patience} exhausted)")
                save_ckpt(model, optimizer, scaler, step+1, best_val,
                          CKPT_DIR / f"early_stop_{step+1}.pt", cfg=CFG)
                break
        model.train()

    if (step + 1) % CFG.save_interval == 0:
        save_ckpt(model, optimizer, scaler, step+1, best_val,
                  CKPT_DIR / f"step_{step+1}.pt", cfg=CFG)
else:
    # Ran to max_steps without early stop
    save_ckpt(model, optimizer, scaler, CFG.max_steps, best_val, CKPT_DIR/"final.pt", cfg=CFG)

train_elapsed = time.time() - train_start
train_hours = train_elapsed / 3600
train_tokens = (step + 1 - start_step) * CFG.eff_batch * CFG.max_seq_len

print(f"\n✓ Training complete. Best val loss: {best_val:.4f}")
print(f"  Total training time: {train_elapsed:.0f}s ({train_hours:.2f} hours)")
print(f"  Steps completed: {step + 1 - start_step:,}")
print(f"  Tokens processed: {train_tokens:,} ({train_tokens/1e6:.0f}M)")

# ── Save metrics JSON ──
metrics["train_time_sec"] = round(train_elapsed, 1)
metrics["train_time_hrs"] = round(train_hours, 3)
metrics["total_tokens"] = train_tokens
metrics["best_val_loss"] = round(best_val, 4)
metrics["experiment"] = "palm"
metrics_path = OUT / "palm_training_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"  Metrics saved to {metrics_path}")

## 8 — Training metrics plots

Visualize training dynamics for your report. Metrics also saved as `training_metrics.json`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("PaLM-style Training Metrics", fontsize=16, fontweight="bold")

# 1. Training Loss
ax = axes[0, 0]
ax.plot(metrics["steps"], metrics["train_loss"], color="steelblue", linewidth=0.8, alpha=0.7)
# Smoothed line (rolling avg over 10 points)
if len(metrics["train_loss"]) > 10:
    w = 10
    smoothed = np.convolve(metrics["train_loss"], np.ones(w)/w, mode="valid")
    ax.plot(metrics["steps"][w-1:], smoothed, color="darkblue", linewidth=2, label="smoothed")
    ax.legend()
ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.set_title("Training Loss")
ax.grid(True, alpha=0.3)

# 2. Validation Loss
ax = axes[0, 1]
if metrics["val_steps"]:
    ax.plot(metrics["val_steps"], metrics["val_loss"], "o-", color="crimson", linewidth=2, markersize=6)
    best_idx = np.argmin(metrics["val_loss"])
    ax.annotate(f'best={metrics["val_loss"][best_idx]:.4f}',
                xy=(metrics["val_steps"][best_idx], metrics["val_loss"][best_idx]),
                fontsize=9, color="darkred", fontweight="bold",
                xytext=(10, 10), textcoords="offset points",
                arrowprops=dict(arrowstyle="->", color="darkred"))
ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.set_title("Validation Loss")
ax.grid(True, alpha=0.3)

# 3. Perplexity (val)
ax = axes[0, 2]
if metrics["val_steps"]:
    ax.plot(metrics["val_steps"], metrics["val_ppl"], "s-", color="darkorange", linewidth=2, markersize=6)
    ax.set_ylabel("Perplexity")
ax.set_xlabel("Step"); ax.set_title("Validation Perplexity")
ax.grid(True, alpha=0.3)

# 4. Learning Rate Schedule
ax = axes[1, 0]
ax.plot(metrics["steps"], metrics["learning_rate"], color="green", linewidth=1.5)
ax.set_xlabel("Step"); ax.set_ylabel("LR"); ax.set_title("Learning Rate Schedule")
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1e"))
ax.grid(True, alpha=0.3)

# 5. Gradient Norm
ax = axes[1, 1]
ax.plot(metrics["steps"], metrics["grad_norm"], color="purple", linewidth=0.8, alpha=0.7)
if len(metrics["grad_norm"]) > 10:
    w = 10
    smoothed_gn = np.convolve(metrics["grad_norm"], np.ones(w)/w, mode="valid")
    ax.plot(metrics["steps"][w-1:], smoothed_gn, color="indigo", linewidth=2, label="smoothed")
    ax.legend()
ax.set_xlabel("Step"); ax.set_ylabel("Grad Norm"); ax.set_title("Gradient Norm")
ax.grid(True, alpha=0.3)

# 6. Throughput (tokens/sec)
ax = axes[1, 2]
ax.plot(metrics["steps"], metrics["tok_per_sec"], color="teal", linewidth=0.8, alpha=0.7)
if metrics["tok_per_sec"]:
    avg_tps = np.mean(metrics["tok_per_sec"])
    ax.axhline(y=avg_tps, color="darkcyan", linestyle="--", linewidth=1.5, label=f"avg={avg_tps:.0f}")
    ax.legend()
ax.set_xlabel("Step"); ax.set_ylabel("Tokens/sec"); ax.set_title("Training Throughput")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / "palm_training_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plots saved to {OUT / 'palm_training_plots.png'}")

# ── Summary table ──
print(f"\n{'='*50}")
print(f"  Experiment:       PaLM-style Parallel Decoder")
print(f"  Training time:    {metrics.get('train_time_hrs', 0):.2f} hours")
print(f"  Total tokens:     {metrics.get('total_tokens', 0)/1e6:.0f}M")
print(f"  Best val loss:    {metrics.get('best_val_loss', 0):.4f}")
if metrics["val_ppl"]:
    print(f"  Best val ppl:     {min(metrics['val_ppl']):.1f}")
if metrics["train_loss"]:
    print(f"  Final train loss: {metrics['train_loss'][-1]:.4f}")
if metrics["tok_per_sec"]:
    print(f"  Avg throughput:   {np.mean(metrics['tok_per_sec']):.0f} tok/s")
print(f"{'='*50}")

In [ ]:
# ── List what's in the checkpoint directory ──
print("Checkpoints on disk:")
if CKPT_DIR.exists():
    for f in sorted(CKPT_DIR.iterdir()):
        mb = f.stat().st_size / 1e6
        print(f"  {f.name:30s} {mb:.1f} MB")
else:
    print("  ⚠ Checkpoint directory does not exist!")

# ── Load best available checkpoint ──
raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
best_path = CKPT_DIR / "best.pt"
if not best_path.exists():
    # Fall back: final.pt → latest step_*.pt → latest early_stop_*.pt
    for pattern in ["final.pt", "step_*.pt", "early_stop_*.pt"]:
        candidates = sorted(CKPT_DIR.glob(pattern),
                            key=lambda p: p.stat().st_mtime, reverse=True)
        if candidates:
            best_path = candidates[0]
            print(f"best.pt not found, falling back to {best_path.name}")
            break

if best_path.exists():
    ckpt = torch.load(best_path, map_location=DEVICE)
    raw_model.load_state_dict(ckpt["model"])
    print(f"Loaded checkpoint: {best_path.name} (step {ckpt['step']}, val_loss={ckpt['val_loss']:.4f})")
else:
    print("⚠ No checkpoint found at all — using current weights from training")

## 9 — Generate predictions for Codabench

In [ ]:
def generate_continuation(prompt, max_words=128):
    ids = tokenizer.encode(prompt)
    max_prompt = CFG.max_seq_len - 200
    if len(ids) > max_prompt: ids = ids[-max_prompt:]
    inp = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    out = model.generate(inp, max_new=max_words * 3,
                         temp=CFG.gen_temperature, top_k=CFG.gen_top_k,
                         top_p=CFG.gen_top_p, rep_pen=CFG.gen_rep_penalty)
    new_ids = out[0, len(ids):].tolist()
    text = tokenizer.decode(new_ids)
    words = text.split()[:max_words]
    return " ".join(words)

# Load test
with open(test_path, "r") as f:
    test_data = json.load(f)

predictions = {}
total = len(test_data)
print(f"Generating continuations for {total} test prompts...")

model.eval()
for i, (tid, item) in enumerate(test_data.items()):
    prompt = item["text"] if isinstance(item, dict) else item
    cont = generate_continuation(prompt, max_words=CFG.gen_max_words)
    predictions[tid] = {"text": cont}
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{total}")

# ── Sanity checks on predictions ──
assert len(predictions) == total, f"Missing predictions: {len(predictions)} vs {total}"
for tid, p in predictions.items():
    assert "text" in p, f"Missing 'text' for {tid}"
    assert len(p["text"].strip()) > 0, f"Empty text for {tid}"
word_counts = [len(p["text"].split()) for p in predictions.values()]
print(f"\n✓ All {total} predictions generated")
print(f"  Words: min={min(word_counts)}, max={max(word_counts)}, "
      f"mean={sum(word_counts)/len(word_counts):.1f}")

# ── Save ──
pred_path = OUT / "palm_prediction.json"
with open(pred_path, "w") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

zip_path = OUT / "palm_submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, "prediction.json")

print(f"  {pred_path}")
print(f"  {zip_path}")

## 10 — Local metrics (BERTScore F1, ROUGE-L, BLEU-4)

Computes metrics on 20 training samples (prompt → continuation) so you can
gauge quality before Codabench submission.

In [ ]:
!pip -q install rouge-score bert-score
import nltk
nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Use test predictions for a quick sanity check on generation quality
eval_gen = [p["text"] for p in list(predictions.values())[:20]]
print(f"Sample generations (first 200 chars):")
for i in range(min(3, len(eval_gen))):
    print(f"  [{i}] {eval_gen[i][:200]}...")
print(f"\n✓ Generation quality check complete. Upload submission.zip to Codabench for real metrics.")

## 11 — Output summary

| File | Description |
|---|---|
| `checkpoints/best.pt` | Best model checkpoint (lowest val loss) |
| `checkpoints/final.pt` | Final model checkpoint |
| `prediction.json` | Codabench predictions (128 words each) |
| `submission.zip` | Ready-to-submit archive |

Upload `submission.zip` to Codabench for evaluation.